# Розділ 7: Створення чат-додатків
## Швидкий старт з API моделей Microsoft Foundry

Ця записна книжка адаптована з [репозиторію зразків Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst), який містить записні книжки, що звертаються до сервісів [Azure OpenAI](notebook-azure-openai.ipynb).

> **Примітка:** GitHub Models буде припинено наприкінці липня 2026 року. Ця записна книжка тепер орієнтована на [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), який пропонує той самий каталог моделей із безкоштовною можливістю спробувати та досвід роботи з Azure AI Inference SDK.


# Огляд  
"Великі мовні моделі — це функції, які відображують текст у текст. Отримавши вхідний рядок тексту, велика мовна модель намагається передбачити, який текст буде далі"(1). Цей блокнот "швидкого старту" ознайомить користувачів з основними поняттями LLM, основними вимогами пакета для початку роботи з AML, м’яким вступом у дизайн підказок і кількома короткими прикладами різних застосувань. 


## Зміст  

[Огляд](#overview)  
[Як користуватися сервісом OpenAI](#how-to-use-openai-service)  
[1. Створення вашого сервісу OpenAI](#1.-creating-your-openai-service)  
[2. Встановлення](#2.-installation)    
[3. Облікові дані](#3.-credentials)  

[Випадки використання](#use-cases)    
[1. Підсумовування тексту](#1.-summarize-text)  
[2. Класифікація тексту](#2.-classify-text)  
[3. Генерація нових назв продуктів](#3.-generate-new-product-names)  
[4. Тонке налаштування класифікатора](#4.fine-tune-a-classifier)  

[Посилання](#references)


### Створіть свій перший запит  
Це коротке завдання надасть базове введення для надсилання запитів до моделі в Microsoft Foundry Models для простої задачі "узагальнення".  


**Кроки**:  
1. Встановіть бібліотеку `azure-ai-inference` у ваше середовище Python, якщо ви цього ще не зробили.  
2. Завантажте стандартні допоміжні бібліотеки та налаштуйте свої облікові дані Microsoft Foundry Models.  
3. Виберіть модель для вашого завдання  
4. Створіть простий запит для моделі  
5. Надішліть свій запит через API моделі!  


### 1. Встановіть `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. Імпортуйте допоміжні бібліотеки та створіть облікові дані


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. Пошук потрібної моделі  
Моделі, такі як GPT-4o і GPT-4o mini, можуть розуміти та генерувати природну мову, і доступні в каталозі Microsoft Foundry Models разом із моделями від Meta, Mistral, Cohere та Microsoft.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-5-mini"


## 4. Проєктування підказок  

"Чарівність великих мовних моделей полягає в тому, що, навчаючись мінімізувати цю помилку передбачення на величезній кількості тексту, моделі з часом опановують концепції, корисні для цих передбачень. Наприклад, вони вивчають такі концепції"(1):

* як писати правильно
* як працює граматика
* як перефразовувати
* як відповідати на запитання
* як підтримувати розмову
* як писати багатьма мовами
* як програмувати
* тощо.

#### Як керувати великою мовною моделлю  
"З усіх вхідних даних для великої мовної моделі найбільший вплив має текстова підказка (prompt)(1).

Великі мовні моделі можна викликати для створення виходу кількома способами:

Інструкція: Скажіть моделі, що ви хочете
Доповнення: Викличте модель завершити початок того, що ви хочете
Демонстрація: Покажіть моделі, що ви хочете, за допомогою:
Кількох прикладів у підказці
Багатьох сотень або тисяч прикладів у навчальному наборі для тонкого налаштування"



#### Існують три базові правила створення підказок:

**Показати і сказати**. Чітко вкажіть, що ви хочете, через інструкції, приклади або їх поєднання. Якщо ви хочете, щоб модель ранжувала список елементів в алфавітному порядку або класифікувала абзац за настроєм, покажіть, що саме ви хочете.

**Надайте якісні дані**. Якщо ви намагаєтесь створити класифікатор або домогтися, щоб модель слідувала шаблону, переконайтеся, що прикладів достатньо. Обов’язково перевірте свої приклади — модель зазвичай досить розумна, аби помітити базові орфографічні помилки і дати вам відповідь, але також може прийняти це як навмисне і це вплине на відповідь.

**Перевірте налаштування.** Параметри temperature та top_p контролюють, наскільки детермінованою є відповідь моделі. Якщо ви хочете відповідь, де є лише одна правильна відповідь, встановіть ці параметри нижчими. Якщо ж хочете отримати більш різноманітні відповіді, встановіть їх вищими. Найпоширеніша помилка користувачів щодо цих налаштувань — вважати їх контролем «розумності» або «креативності».


Джерело: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Надіслати!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### Повторіть той самий виклик, як порівнюються результати? 


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## Підсумуйте текст  
#### Виклик  
Підсумуйте текст, додавши 'tl;dr:' в кінець текстового фрагмента. Зверніть увагу, як модель розуміє, як виконувати низку завдань без додаткових інструкцій. Ви можете експериментувати з більш описовими підказками, ніж tl;dr, щоб змінити поведінку моделі та налаштувати підсумок, який отримаєте(3).  

Останні дослідження показали значні покращення в багатьох завданнях та бенчмарках NLP шляхом початкового тренування на великому корпусі текстів, а потім донавчання для конкретного завдання. Незважаючи на те, що архітектура зазвичай не залежить від завдання, цей метод все одно вимагає спеціалізованих наборів даних для донавчання з тисячами або десятками тисяч прикладів. Навпаки, люди загалом можуть виконувати нове мовне завдання лише за кілька прикладів або простих інструкцій — те, з чим сучасні системи NLP все ще значною мірою борються. Тут ми показуємо, що масштабування мовних моделей значно покращує постановку завдань незалежно від конкретного завдання з небагатьма прикладами, іноді навіть досягаючи конкурентоспроможності з попередніми передовими методами донавчання.  



Tl;dr


# Вправи для кількох випадків використання  
1. Резюмувати текст  
2. Класифікувати текст  
3. Генерувати нові назви продуктів


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Класифікувати текст  
#### Виклик  
Класифікуйте елементи за категоріями, наданими в момент виконання. У наступному прикладі ми надаємо як категорії, так і текст для класифікації у підказці (*playground_reference). 

Запит клієнта: Вітаю, нещодавно зламалась одна з клавіш на клавіатурі мого ноутбука, і мені потрібна заміна:

Класифікована категорія:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Генеруйте нові назви продуктів
#### Виклик
Створюйте назви продуктів на основі прикладів слів. Тут ми включаємо в підказку інформацію про продукт, для якого ми збираємося створювати назви. Ми також надаємо схожий приклад, щоб показати бажаний патерн. Крім того, ми встановили високе значення температури для збільшення випадковості та більш інноваційних відповідей.

Опис продукту: Пристрій для приготування молочних коктейлів вдома
Початкові слова: швидкий, здоровий, компактний.
Назви продуктів: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Опис продукту: Пара взуття, яка підходить для будь-якого розміру ноги.
Початкові слова: адаптивний, підходить, універсальний.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# Джерела  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry портал](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Кращі практики тонкого налаштування моделей GPT для класифікації тексту](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Для додаткової допомоги  
[Команда комерціалізації OpenAI](AzureOpenAITeam@microsoft.com) 


# Учасники
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу штучного інтелекту для перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний людський переклад. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
